# IMSFD — InceptionResnetV1 Partial Fine-Tuning + ArcFace (v2)

Notebook ini disesuaikan dengan **struktur asli dataset IMSFD** yang Anda jelaskan:

```text
IMSFD/
├── A1/
│   ├── Training/
│   │   ├── NIM_001/  → foto JPG/PNG
│   │   └── ...
│   └── Testing/
│       ├── NIM_001/  → foto JPG/PNG
│       └── ...
├── A2/
│   ├── Training/...
│   └── Testing/...
└── B1/
    ├── Training/...
    └── Testing/...
```

**Perbaikan penting dari notebook lama:** identitas diambil dari folder NIM tepat di bawah `Training` atau `Testing`, bukan dari folder `A1`, `A2`, atau `B1`.

## Keputusan pengolahan data

1. Folder asli IMSFD **tidak dipindahkan, diganti nama, atau diduplikasi**.
2. Notebook membuat manifest yang menyatukan identitas dari A1, A2, dan B1, tetapi tetap menyimpan asal subset dan split resmi.
3. Folder resmi `Training` digunakan untuk training. Sebagian kecil dari dalam `Training` disisihkan untuk validation.
4. Folder resmi `Testing` tetap disimpan sebagai test final dan tidak dipakai untuk mengubah bobot atau memilih threshold.
5. Label internal model dibuat berurutan `ID_001`, `ID_002`, dan seterusnya. Pemetaan ke nama folder NIM disimpan terpisah di Google Drive.

**Jangan menghapus folder output lama `IMSFD_ArcFace_Output`.** Notebook ini memakai folder output baru `IMSFD_ArcFace_Output_v2` agar hasil audit lama tetap aman.

> **Cara menjalankan:** jalankan cell berurutan. Setelah bagian audit dan visualisasi contoh gambar selesai, periksa hasilnya terlebih dahulu. Jangan lanjut ke training sebelum struktur dan sampel wajah dipastikan benar.

> Notebook ini mengasumsikan gambar sudah berupa *face crop*. Belum ada deteksi/alignment wajah otomatis. Pengolahan input mengikuti ukuran 160×160 dan normalisasi yang digunakan `facenet-pytorch`.

## 1. Instalasi library dan pemeriksaan runtime

In [ ]:
!pip -q install facenet-pytorch scikit-learn pandas matplotlib tqdm

import os, sys, json, math, time, random, shutil, warnings
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1

from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
if USE_AMP:
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('PERINGATAN: GPU tidak terdeteksi. Untuk Colab, pilih Runtime > Change runtime type > T4 GPU (jika tersedia).')

## 2. Hubungkan Google Drive dan tentukan lokasi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MYDRIVE = Path('/content/drive/MyDrive')

# Notebook mencoba dua variasi huruf nama folder. Ubah manual jika nama folder Anda berbeda.
DATA_ROOT_CANDIDATES = [MYDRIVE / 'IMSFD', MYDRIVE / 'imsfd']
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.exists()), DATA_ROOT_CANDIDATES[0])

# Folder baru: tidak menimpa IMSFD_ArcFace_Output yang sudah Anda punya.
OUTPUT_DIR = MYDRIVE / 'IMSFD_ArcFace_Output_v2'
AUDIT_DIR = OUTPUT_DIR / '01_audit'
SPLIT_DIR = OUTPUT_DIR / '02_splits'
CHECKPOINT_DIR = OUTPUT_DIR / '03_checkpoints'
PLOT_DIR = OUTPUT_DIR / '04_plots'
EXPORT_DIR = OUTPUT_DIR / '05_model_export'

for folder in [OUTPUT_DIR, AUDIT_DIR, SPLIT_DIR, CHECKPOINT_DIR, PLOT_DIR, EXPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}
SUBSET_NAMES = {'a1', 'a2', 'b1'}
SPLIT_NAMES = {'training', 'testing'}
IMAGE_SIZE = 160
EXPECTED_IDENTITIES = 68  # Hanya untuk pemeriksaan/peringatan, bukan membatasi data.

print('DATA_ROOT :', DATA_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('DATA_ROOT exists:', DATA_ROOT.exists())
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f'Folder dataset tidak ditemukan: {DATA_ROOT}. Periksa nama dan lokasi folder IMSFD di MyDrive.'
    )

## 3. Temukan folder identitas dari struktur IMSFD asli

Langkah ini hanya mengidentifikasi folder `A1/A2/B1 → Training/Testing → NIM`. Belum membaca semua gambar.

Contoh: untuk file `IMSFD/A1/Training/12345678/foto01.jpg`, notebook mengambil:
- `subset = A1`
- `official_split = Training`
- `identity = 12345678`

Jika struktur sebenarnya berbeda, notebook akan berhenti dengan pesan yang menjelaskan apa yang tidak ditemukan.

In [ ]:
identity_dirs = []
structure_warnings = []

subset_dirs = sorted(
    [p for p in DATA_ROOT.iterdir() if p.is_dir() and p.name.lower() in SUBSET_NAMES],
    key=lambda p: p.name.lower()
)

if not subset_dirs:
    visible_dirs = [p.name for p in DATA_ROOT.iterdir() if p.is_dir()]
    raise RuntimeError(
        'Tidak menemukan folder A1, A2, atau B1 langsung di DATA_ROOT. '
        f'Folder yang terlihat: {visible_dirs[:30]}. Pastikan DATA_ROOT menunjuk ke folder IMSFD yang benar.'
    )

for subset_dir in subset_dirs:
    split_dirs = {
        p.name.lower(): p for p in subset_dir.iterdir()
        if p.is_dir() and p.name.lower() in SPLIT_NAMES
    }
    for split_name in ['training', 'testing']:
        split_dir = split_dirs.get(split_name)
        if split_dir is None:
            structure_warnings.append(f'{subset_dir.name}: folder {split_name} tidak ditemukan')
            continue
        for identity_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
            identity_dirs.append({
                'subset': subset_dir.name.upper(),
                'official_split': 'Training' if split_name == 'training' else 'Testing',
                'identity': identity_dir.name,
                'identity_dir': identity_dir,
            })

if not identity_dirs:
    raise RuntimeError('Folder identitas NIM tidak ditemukan di dalam A1/A2/B1 → Training/Testing.')

identity_dir_table = pd.DataFrame([
    {k: str(v) if k == 'identity_dir' else v for k, v in row.items()}
    for row in identity_dirs
])
print('Folder subset ditemukan:', [p.name for p in subset_dirs])
print('Folder identitas (gabungan Training dan Testing):', len(identity_dirs))
print('Jumlah identitas unik:', identity_dir_table['identity'].nunique())
print('\nJumlah folder identitas per subset dan split:')
display(identity_dir_table.groupby(['subset', 'official_split']).agg(
    identity_folders=('identity', 'nunique')
).reset_index())
if structure_warnings:
    print('\nPERINGATAN STRUKTUR:')
    for msg in structure_warnings:
        print('-', msg)

## 4. Audit gambar — progress bar dan penyimpanan berkala

Audit memeriksa JPG/JPEG/PNG, file yang rusak, dimensi, mode warna, ukuran file, subset, split resmi, dan identitas NIM.

- Progress ditampilkan per folder identitas.
- Setiap 5 folder identitas, checkpoint sementara disimpan ke `01_audit`.
- Jika proses terhenti, file checkpoint sementara tidak dihapus.
- Saat audit selesai penuh, checkpoint dihapus dan `dataset_manifest.csv` menjadi hasil final.

In [ ]:
records = []
corrupt_files = []
unsupported_files = []
empty_identity_dirs = []
checkpoint_path = AUDIT_DIR / 'dataset_manifest_SCAN_CHECKPOINT.csv'
final_manifest_path = AUDIT_DIR / 'dataset_manifest.csv'


def atomic_save_csv(rows, destination):
    """Simpan CSV melalui file sementara agar checkpoint tidak setengah tertulis."""
    tmp_path = destination.with_name(destination.stem + '.tmp.csv')
    pd.DataFrame(rows).to_csv(tmp_path, index=False)
    os.replace(tmp_path, destination)


try:
    for folder_no, info in enumerate(tqdm(identity_dirs, desc='Audit folder identitas'), start=1):
        identity_dir = info['identity_dir']
        image_paths = sorted([
            p for p in identity_dir.rglob('*')
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        ])
        if not image_paths:
            empty_identity_dirs.append({
                'subset': info['subset'],
                'official_split': info['official_split'],
                'identity': info['identity'],
                'identity_dir': str(identity_dir),
            })

        for path in image_paths:
            try:
                with Image.open(path) as im:
                    im.verify()
                with Image.open(path) as im:
                    width, height = im.size
                    mode = im.mode
                    image_format = im.format
                records.append({
                    'path': str(path),
                    'relative_path': str(path.relative_to(DATA_ROOT)),
                    'subset': info['subset'],
                    'official_split': info['official_split'],
                    'identity': info['identity'],
                    'filename': path.name,
                    'width': int(width),
                    'height': int(height),
                    'mode': mode,
                    'format': image_format,
                    'ext': path.suffix.lower(),
                    'file_size_bytes': int(path.stat().st_size),
                })
            except Exception as exc:
                corrupt_files.append({
                    'path': str(path),
                    'subset': info['subset'],
                    'official_split': info['official_split'],
                    'identity': info['identity'],
                    'error': repr(exc),
                })

        # Simpan checkpoint setiap lima folder agar aman bila runtime terputus.
        if folder_no % 5 == 0:
            atomic_save_csv(records, checkpoint_path)
            tqdm.write(f'Checkpoint tersimpan: {folder_no}/{len(identity_dirs)} folder | gambar valid: {len(records):,}')

except KeyboardInterrupt:
    atomic_save_csv(records, checkpoint_path)
    pd.DataFrame(corrupt_files).to_csv(AUDIT_DIR / 'corrupt_files_PARTIAL.csv', index=False)
    print('\nAudit dihentikan manual. Checkpoint parsial tetap disimpan di:', checkpoint_path)
    raise

manifest = pd.DataFrame(records)
if manifest.empty:
    raise RuntimeError('Tidak ditemukan gambar JPG/JPEG/PNG valid. Periksa lokasi dan struktur dataset.')

manifest = manifest.sort_values(['identity', 'official_split', 'path']).reset_index(drop=True)
manifest.to_csv(final_manifest_path, index=False)
if checkpoint_path.exists():
    checkpoint_path.unlink()

if corrupt_files:
    pd.DataFrame(corrupt_files).to_csv(AUDIT_DIR / 'corrupt_files.csv', index=False)
if unsupported_files:
    pd.DataFrame(unsupported_files).to_csv(AUDIT_DIR / 'unsupported_files.csv', index=False)
if empty_identity_dirs:
    pd.DataFrame(empty_identity_dirs).to_csv(AUDIT_DIR / 'empty_identity_folders.csv', index=False)

print('Audit selesai.')
print('Manifest:', final_manifest_path)
print('Gambar valid:', f'{len(manifest):,}')
print('File rusak:', len(corrupt_files))

## 5. Ringkasan audit — jumlah identitas, split, subset, format, dan resolusi

In [ ]:
# Satu identitas dapat muncul pada Training dan Testing. Identitas unik dihitung dari nama folder NIM.
identity_counts = manifest.pivot_table(
    index='identity', columns='official_split', values='path', aggfunc='count', fill_value=0
)
for col in ['Training', 'Testing']:
    if col not in identity_counts.columns:
        identity_counts[col] = 0
identity_counts = identity_counts[['Training', 'Testing']]
identity_counts['Total'] = identity_counts['Training'] + identity_counts['Testing']
identity_counts = identity_counts.reset_index()

identity_subset = manifest.groupby('identity')['subset'].agg(lambda x: ','.join(sorted(set(x)))).rename('subset')
identity_counts = identity_counts.merge(identity_subset, on='identity', how='left')
identity_counts = identity_counts[['identity', 'subset', 'Training', 'Testing', 'Total']]
identity_counts.to_csv(AUDIT_DIR / 'identity_counts_by_split.csv', index=False)

subset_summary = manifest.groupby(['subset', 'official_split']).agg(
    unique_identities=('identity', 'nunique'),
    valid_images=('path', 'count')
).reset_index()
subset_summary.to_csv(AUDIT_DIR / 'subset_split_summary.csv', index=False)

format_summary = manifest.groupby(['ext']).size().rename('images').reset_index()
format_summary.to_csv(AUDIT_DIR / 'format_summary.csv', index=False)
mode_summary = manifest.groupby(['mode']).size().rename('images').reset_index()
mode_summary.to_csv(AUDIT_DIR / 'color_mode_summary.csv', index=False)
resolution_summary = manifest.groupby(['width', 'height']).size().rename('images').reset_index()
resolution_summary = resolution_summary.sort_values('images', ascending=False)
resolution_summary.to_csv(AUDIT_DIR / 'resolution_summary.csv', index=False)

multi_subset = manifest.groupby('identity')['subset'].nunique()
identities_multiple_subsets = multi_subset[multi_subset > 1].index.tolist()
if identities_multiple_subsets:
    manifest[manifest['identity'].isin(identities_multiple_subsets)].sort_values(['identity','subset']).to_csv(
        AUDIT_DIR / 'identities_appearing_in_multiple_subsets.csv', index=False
    )

summary = {
    'dataset_root': str(DATA_ROOT),
    'valid_images': int(len(manifest)),
    'corrupt_images': int(len(corrupt_files)),
    'unique_identities': int(manifest['identity'].nunique()),
    'expected_identities': int(EXPECTED_IDENTITIES),
    'training_images': int((manifest['official_split'] == 'Training').sum()),
    'testing_images': int((manifest['official_split'] == 'Testing').sum()),
    'identity_folders_without_valid_images': int(len(empty_identity_dirs)),
    'identities_present_in_multiple_subsets': identities_multiple_subsets,
}
with open(AUDIT_DIR / 'audit_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('RINGKASAN DATASET')
for key, value in summary.items():
    print(f'{key}: {value}')
print('\nRINGKASAN PER SUBSET DAN SPLIT')
display(subset_summary)
print('\nFORMAT GAMBAR')
display(format_summary)
print('\nRESOLUSI PALING UMUM')
display(resolution_summary.head(15))
print('\nJUMLAH GAMBAR PER IDENTITAS (10 identitas dengan gambar paling sedikit)')
display(identity_counts.sort_values('Total').head(10))

if manifest['identity'].nunique() != EXPECTED_IDENTITIES:
    print(f'CATATAN: ditemukan {manifest["identity"].nunique()} identitas unik, sedangkan perkiraan Anda {EXPECTED_IDENTITIES}. Periksa hasil audit dan struktur folder.')
if identities_multiple_subsets:
    print('PERHATIAN: ada identitas dengan nama folder sama pada lebih dari satu subset. Periksa file identities_appearing_in_multiple_subsets.csv.')

## 6. Lihat contoh foto dari Training

Periksa apakah gambar benar-benar berisi satu wajah, wajah terlihat jelas, tidak salah identitas, dan sudah ter-crop secara konsisten. Contoh diambil dari beberapa identitas berbeda, bukan hanya satu folder.

In [ ]:
train_manifest_audit = manifest[manifest['official_split'] == 'Training'].copy()
if train_manifest_audit.empty:
    raise RuntimeError('Tidak ada gambar valid pada split Training.')

available_ids = sorted(train_manifest_audit['identity'].unique())
rng_sample = np.random.default_rng(SEED)
chosen_ids = rng_sample.choice(available_ids, size=min(12, len(available_ids)), replace=False)
sample_rows = []
for identity in chosen_ids:
    candidates = train_manifest_audit[train_manifest_audit['identity'] == identity]
    sample_rows.append(candidates.sample(1, random_state=SEED).iloc[0])

fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for ax in axes.flat:
    ax.axis('off')
for ax, row in zip(axes.flat, sample_rows):
    with Image.open(row['path']) as im:
        ax.imshow(ImageOps.exif_transpose(im).convert('RGB'))
    ax.set_title(f"{row['identity']} | {row['subset']}", fontsize=8)
plt.tight_layout()
plt.savefig(AUDIT_DIR / 'sample_training_faces.png', dpi=160, bbox_inches='tight')
plt.show()

print('SIMPANAN AUDIT:', AUDIT_DIR)
print('PENTING: sebelum melanjutkan, pastikan folder identitas dan contoh gambar sudah benar.')

---

# BAGIAN PELATIHAN MODEL

**Jangan lanjutkan bagian ini sebelum audit dan contoh gambar diperiksa.**

## 7. Menyusun training, validation, dan test manifest

Kita mempertahankan pembagian resmi IMSFD:
- Semua gambar pada folder resmi `Testing` menjadi test final.
- Folder resmi `Training` dibagi lagi menjadi 85% training dan 15% validation per identitas.
- Validation tidak mengubah bobot; test tidak dipakai untuk memilih hyperparameter atau threshold.

Dengan demikian, kita tidak mencampur gambar dari folder `Testing` ke proses pelatihan.

In [ ]:
VAL_RATIO = 0.15

train_pool = manifest[manifest['official_split'] == 'Training'].copy()
test_official = manifest[manifest['official_split'] == 'Testing'].copy()

if train_pool.empty:
    raise RuntimeError('Manifest Training kosong.')
if test_official.empty:
    raise RuntimeError('Manifest Testing kosong. Periksa apakah folder Testing berhasil terbaca.')

classes = sorted(train_pool['identity'].unique().tolist())
class_to_idx = {identity: idx for idx, identity in enumerate(classes)}

train_rows, val_rows = [], []
insufficient_train = []
rng = np.random.default_rng(SEED)

for identity in classes:
    group = train_pool[train_pool['identity'] == identity].sample(frac=1, random_state=SEED).reset_index(drop=True)
    n = len(group)
    if n < 4:
        insufficient_train.append((identity, n))
        continue
    n_val = max(2, int(round(n * VAL_RATIO)))  # minimal 2 foto agar bisa membentuk genuine pair
    n_val = min(n_val, n - 2)                  # sisakan minimal 2 foto untuk training
    val_rows.extend(group.iloc[:n_val].to_dict('records'))
    train_rows.extend(group.iloc[n_val:].to_dict('records'))

if insufficient_train:
    raise ValueError(
        'Ada identitas dengan kurang dari 4 gambar valid di Training sehingga pembagian train/validation tidak aman: '
        + str(insufficient_train[:20])
    )

train_df = pd.DataFrame(train_rows).reset_index(drop=True)
val_df = pd.DataFrame(val_rows).reset_index(drop=True)

# Test hanya mencakup identitas yang dikenal saat training dan memiliki >=2 foto Testing.
test_counts = test_official.groupby('identity').size()
test_eligible_ids = sorted([
    identity for identity in classes
    if identity in test_counts.index and int(test_counts.loc[identity]) >= 2
])
test_not_eligible = sorted([identity for identity in classes if identity not in test_eligible_ids])
test_df = test_official[test_official['identity'].isin(test_eligible_ids)].copy().reset_index(drop=True)

if len(test_eligible_ids) < 2:
    raise RuntimeError('Kurang dari dua identitas memiliki minimal dua foto pada Testing. Pasangan genuine/impostor belum dapat dihitung.')

for df in [train_df, val_df, test_df]:
    df['class_index'] = df['identity'].map(class_to_idx).astype(int)

train_df.to_csv(SPLIT_DIR / 'train_manifest.csv', index=False)
val_df.to_csv(SPLIT_DIR / 'validation_manifest.csv', index=False)
test_df.to_csv(SPLIT_DIR / 'test_manifest_used.csv', index=False)

mapping = pd.DataFrame({
    'class_index': list(range(len(classes))),
    'model_label': [f'ID_{i+1:03d}' for i in range(len(classes))],
    'original_identity_folder': classes,
})
mapping.to_csv(SPLIT_DIR / 'identity_mapping.csv', index=False)

split_summary = pd.DataFrame([
    {'split': 'train', 'images': len(train_df), 'identities': train_df['identity'].nunique()},
    {'split': 'validation', 'images': len(val_df), 'identities': val_df['identity'].nunique()},
    {'split': 'official_test_used', 'images': len(test_df), 'identities': test_df['identity'].nunique()},
])
print('Jumlah kelas untuk ArcFace:', len(classes))
display(split_summary)
print('Identitas Training dengan data Testing kurang dari 2 foto (tidak masuk evaluasi pasangan):', len(test_not_eligible))
if test_not_eligible:
    print(test_not_eligible[:20])
print('Catatan: semua foto Testing yang tidak memenuhi syarat genuine-pair tetap tercatat dalam manifest audit asli.')

## 8. (Disarankan) Salin data ke penyimpanan sementara Colab

Google Drive cocok untuk menyimpan dataset dan hasil, tetapi membaca puluhan ribu file kecil langsung dari Drive dapat memperlambat DataLoader. Cell ini menyalin hanya gambar yang diperlukan untuk train/validation/test ke `/content` (penyimpanan sementara Colab), tanpa mengubah atau menghapus file asli di Drive. Jika ruang disk tidak cukup, ubah `CACHE_DATA_LOCALLY = False`.

In [ ]:
CACHE_DATA_LOCALLY = True
LOCAL_DATA_ROOT = Path('/content/IMSFD_local_cache')

if CACHE_DATA_LOCALLY:
    required_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    required_df = required_df[['path', 'relative_path', 'file_size_bytes']].drop_duplicates('path')
    required_bytes = int(required_df['file_size_bytes'].sum())
    free_bytes = shutil.disk_usage('/content').free
    print(f'Ukuran data yang akan disalin: {required_bytes / (1024**3):.2f} GB')
    print(f'Ruang kosong /content: {free_bytes / (1024**3):.2f} GB')
    if required_bytes > int(free_bytes * 0.85):
        raise RuntimeError(
            'Ruang penyimpanan Colab tidak cukup dengan buffer 15%. Ubah CACHE_DATA_LOCALLY=False ' 
            'agar DataLoader membaca langsung dari Google Drive (lebih lambat). '
        )

    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    copied, skipped = 0, 0
    for row in tqdm(required_df.itertuples(index=False), total=len(required_df), desc='Menyalin data ke /content'):
        source = Path(row.path)
        destination = LOCAL_DATA_ROOT / Path(row.relative_path)
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.exists() and destination.stat().st_size == int(row.file_size_bytes):
            skipped += 1
        else:
            shutil.copy2(source, destination)
            copied += 1

    for dataframe in [train_df, val_df, test_df]:
        dataframe['path'] = dataframe['relative_path'].apply(lambda value: str(LOCAL_DATA_ROOT / Path(value)))

    print(f'Selesai. File disalin: {copied:,} | sudah tersedia: {skipped:,}')
    print('DataLoader akan membaca data dari:', LOCAL_DATA_ROOT)
else:
    print('DataLoader akan membaca gambar langsung dari Google Drive. Ini dapat lebih lambat.')


## 9. Preprocessing gambar dan DataLoader

- JPG/JPEG/PNG dibaca dan dikonversi ke RGB.
- Orientasi EXIF diterapkan sebelum resize.
- Gambar dipadding menjadi persegi agar rasio wajah tidak diregangkan.
- Ukuran akhir 160×160.
- Normalisasi mengikuti standar input `facenet-pytorch`: `(pixel - 127.5) / 128` setelah piksel berada pada skala 0–255.
- Augmentasi sederhana hanya dipakai pada training.

In [ ]:
class SquarePad:
    def __call__(self, img):
        width, height = img.size
        side = max(width, height)
        left = (side - width) // 2
        top = (side - height) // 2
        right = side - width - left
        bottom = side - height - top
        return ImageOps.expand(img, border=(left, top, right, bottom), fill=(127, 127, 127))

class FixedImageStandardization:
    def __call__(self, tensor):
        return (tensor * 255.0 - 127.5) / 128.0

train_transform = transforms.Compose([
    SquarePad(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.04, hue=0.01),
    transforms.ToTensor(),
    FixedImageStandardization(),
])

eval_transform = transforms.Compose([
    SquarePad(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    FixedImageStandardization(),
])

class IMSFDFaceDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, index):
        row = self.df.iloc[index]
        path = row['path']
        with Image.open(path) as im:
            image = ImageOps.exif_transpose(im).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row['class_index']), path

BATCH_SIZE = 16
NUM_WORKERS = 2

train_ds = IMSFDFaceDataset(train_df, train_transform)
val_ds = IMSFDFaceDataset(val_df, eval_transform)
test_ds = IMSFDFaceDataset(test_df, eval_transform)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=USE_AMP, drop_last=True, persistent_workers=(NUM_WORKERS > 0)
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=USE_AMP, persistent_workers=(NUM_WORKERS > 0)
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=USE_AMP, persistent_workers=(NUM_WORKERS > 0)
)

if len(train_loader) == 0:
    raise RuntimeError('DataLoader training kosong. Turunkan BATCH_SIZE atau periksa jumlah gambar.')

print(f'Train: {len(train_df):,} foto | {len(classes)} identitas')
print(f'Validation: {len(val_df):,} foto | {val_df.identity.nunique()} identitas')
print(f'Test resmi yang dipakai: {len(test_df):,} foto | {test_df.identity.nunique()} identitas')
print('Jumlah batch training per epoch:', len(train_loader))

## 10. Inisialisasi InceptionResnetV1 pretrained VGGFace2 dan ArcFace

Konfigurasi partial fine-tuning awal:
- Frozen: seluruh parameter terlebih dahulu.
- Trainable: `repeat_3`, `block8`, `last_linear`, `last_bn`.
- ArcFace head memiliki satu kelas untuk setiap identitas pada folder Training.
- BatchNorm tetap memakai running statistics pretrained agar statistiknya tidak berubah drastis pada dataset yang lebih kecil; parameter affine di modul yang dibuka tetap dapat dilatih.

In [ ]:
class ArcMarginProduct(nn.Module):
    """ArcFace head: normalized feature/weight + additive angular margin."""
    def __init__(self, in_features, out_features, s=64.0, m=0.50, easy_margin=False):
        super().__init__()
        self.in_features = int(in_features)
        self.out_features = int(out_features)
        self.s = float(s)
        self.m = float(m)
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.easy_margin = easy_margin
        self.cos_m = math.cos(self.m)
        self.sin_m = math.sin(self.m)
        self.th = math.cos(math.pi - self.m)
        self.mm = math.sin(math.pi - self.m) * self.m

    def forward(self, embeddings, labels):
        cosine = F.linear(F.normalize(embeddings.float()), F.normalize(self.weight.float()))
        cosine = cosine.clamp(-1.0, 1.0)
        sine = torch.sqrt(torch.clamp(1.0 - cosine.pow(2), min=1e-7))
        phi = cosine * self.cos_m - sine * self.sin_m
        if self.easy_margin:
            phi = torch.where(cosine > 0, phi, cosine)
        else:
            phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = F.one_hot(labels, num_classes=self.out_features).to(dtype=cosine.dtype)
        logits = one_hot * phi + (1.0 - one_hot) * cosine
        return logits * self.s

num_classes = len(classes)
model = InceptionResnetV1(pretrained='vggface2', classify=False).to(DEVICE)
arcface_head = ArcMarginProduct(512, num_classes, s=64.0, m=0.50).to(DEVICE)

# Bekukan semua bobot backbone.
for parameter in model.parameters():
    parameter.requires_grad = False

# Buka hanya modul akhir yang ditetapkan.
UNFREEZE_MODULES = ['repeat_3', 'block8', 'last_linear', 'last_bn']
for module_name in UNFREEZE_MODULES:
    module = getattr(model, module_name, None)
    if module is None:
        raise AttributeError(f'Modul {module_name} tidak ditemukan pada versi facenet-pytorch yang terpasang.')
    for parameter in module.parameters():
        parameter.requires_grad = True

backbone_trainable = [p for p in model.parameters() if p.requires_grad]
head_trainable = list(arcface_head.parameters())
num_trainable = sum(p.numel() for p in backbone_trainable) + sum(p.numel() for p in head_trainable)
num_total = sum(p.numel() for p in model.parameters()) + sum(p.numel() for p in arcface_head.parameters())
print('Unfreeze modules:', UNFREEZE_MODULES)
print(f'Trainable parameters: {num_trainable:,} / {num_total:,} ({100*num_trainable/num_total:.2f}%)')

BACKBONE_LR = 1e-5
ARCFACE_LR = 1e-3
WEIGHT_DECAY = 1e-4
optimizer = torch.optim.AdamW([
    {'params': backbone_trainable, 'lr': BACKBONE_LR},
    {'params': head_trainable, 'lr': ARCFACE_LR},
], weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)
criterion = nn.CrossEntropyLoss()
try:
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print('Model dan ArcFace siap.')

## 11. Pelatihan partial fine-tuning

Checkpoint `last_checkpoint.pth` dan `best_checkpoint.pth` disimpan setiap epoch. Jika runtime Colab terputus, ubah `RESUME_FROM` menjadi path checkpoint terakhir, lalu jalankan ulang cell training.

Parameter awal ini adalah titik awal yang konservatif, bukan jaminan jumlah epoch atau hasil tertentu. Pantau `train_loss` dan `val_loss`; jika terjadi CUDA out-of-memory, turunkan batch size menjadi 8 lalu buat ulang DataLoader sebelum mengulang training.

In [ ]:
EPOCHS = 15
PATIENCE = 4
RESUME_FROM = None  # Contoh: str(CHECKPOINT_DIR / 'last_checkpoint.pth')

start_epoch = 0
best_val_loss = float('inf')
no_improve = 0
history = []


def safe_torch_load(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)

if RESUME_FROM is not None:
    resume_path = Path(RESUME_FROM)
    if not resume_path.exists():
        raise FileNotFoundError(f'Checkpoint resume tidak ditemukan: {resume_path}')
    checkpoint = safe_torch_load(resume_path)
    model.load_state_dict(checkpoint['model_state'])
    arcface_head.load_state_dict(checkpoint['arcface_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    if 'scheduler_state' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state'])
    if 'scaler_state' in checkpoint:
        scaler.load_state_dict(checkpoint['scaler_state'])
    start_epoch = int(checkpoint['epoch']) + 1
    best_val_loss = float(checkpoint.get('best_val_loss', float('inf')))
    no_improve = int(checkpoint.get('no_improve', 0))
    history = checkpoint.get('history', [])
    print('Melanjutkan dari epoch:', start_epoch + 1)


def autocast_context():
    if USE_AMP:
        return torch.autocast(device_type='cuda', dtype=torch.float16)
    return nullcontext()


def set_training_mode_with_frozen_bn():
    # Dropout aktif, tetapi BatchNorm memakai running statistics pretrained.
    model.train()
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
    arcface_head.train()


def run_epoch(loader, training):
    if training:
        set_training_mode_with_frozen_bn()
    else:
        model.eval()
        arcface_head.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    grad_context = torch.enable_grad() if training else torch.no_grad()

    with grad_context:
        for images, targets, _ in tqdm(loader, leave=False, desc='train' if training else 'validation'):
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)

            with autocast_context():
                embeddings = model(images)

            # ArcFace dihitung dalam FP32 agar operasi sudut lebih stabil.
            logits = arcface_head(embeddings.float(), targets)
            loss = criterion(logits, targets)

            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(backbone_trainable + head_trainable, max_norm=5.0)
                scaler.step(optimizer)
                scaler.update()

            batch_size = targets.size(0)
            total_loss += float(loss.detach().item()) * batch_size
            total_correct += int((logits.detach().argmax(dim=1) == targets).sum().item())
            total_samples += batch_size

    return total_loss / max(total_samples, 1), total_correct / max(total_samples, 1)


for epoch in range(start_epoch, EPOCHS):
    epoch_start = time.time()
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss, val_acc = run_epoch(val_loader, training=False)
    scheduler.step(val_loss)

    row = {
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_arcface_top1': train_acc,
        'val_loss': val_loss,
        'val_arcface_top1': val_acc,
        'backbone_lr': optimizer.param_groups[0]['lr'],
        'arcface_lr': optimizer.param_groups[1]['lr'],
        'seconds': time.time() - epoch_start,
    }
    history.append(row)
    print(row)

    improved = val_loss < best_val_loss
    if improved:
        best_val_loss = val_loss
        no_improve = 0
    else:
        no_improve += 1

    checkpoint = {
        'epoch': epoch,
        'model_state': model.state_dict(),
        'arcface_state': arcface_head.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict(),
        'best_val_loss': best_val_loss,
        'no_improve': no_improve,
        'history': history,
        'classes': classes,
        'class_to_idx': class_to_idx,
        'unfreeze_modules': UNFREEZE_MODULES,
        'image_size': IMAGE_SIZE,
        'seed': SEED,
    }
    torch.save(checkpoint, CHECKPOINT_DIR / 'last_checkpoint.pth')
    if improved:
        torch.save(checkpoint, CHECKPOINT_DIR / 'best_checkpoint.pth')
        print('  -> best checkpoint diperbarui')

    pd.DataFrame(history).to_csv(OUTPUT_DIR / 'training_history.csv', index=False)
    if no_improve >= PATIENCE:
        print('Early stopping: validation loss tidak membaik.')
        break

print('Pelatihan selesai. Best validation loss:', best_val_loss)

## 12. Grafik riwayat pelatihan

In [ ]:
history_df = pd.DataFrame(history)
if history_df.empty:
    raise RuntimeError('Riwayat training kosong. Periksa apakah cell training telah dijalankan.')
display(history_df)

plt.figure(figsize=(8, 4))
plt.plot(history_df['epoch'], history_df['train_loss'], label='Train loss')
plt.plot(history_df['epoch'], history_df['val_loss'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('ArcFace cross-entropy loss')
plt.title('Training dan Validation Loss')
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig(PLOT_DIR / 'loss_curve.png', dpi=160)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history_df['epoch'], history_df['train_arcface_top1'], label='Train ArcFace top-1')
plt.plot(history_df['epoch'], history_df['val_arcface_top1'], label='Validation ArcFace top-1')
plt.xlabel('Epoch')
plt.ylabel('Identity classification accuracy')
plt.title('ArcFace Top-1 Training Metric')
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig(PLOT_DIR / 'arcface_top1_curve.png', dpi=160)
plt.show()

## 13. Muat checkpoint terbaik dan ekspor backbone

Untuk membuat embedding saat verifikasi, kita menggunakan **backbone InceptionResnetV1 hasil fine-tuning**. ArcFace head dipakai saat training, tetapi tidak diperlukan untuk menghitung cosine similarity antara dua embedding.

In [ ]:
best_path = CHECKPOINT_DIR / 'best_checkpoint.pth'
if not best_path.exists():
    raise FileNotFoundError('best_checkpoint.pth tidak ditemukan. Jalankan training sampai minimal satu epoch selesai.')

best_ckpt = safe_torch_load(best_path)
model.load_state_dict(best_ckpt['model_state'])
model.eval()

backbone_export_path = EXPORT_DIR / 'inceptionresnetv1_imsfd_partial_finetuned.pth'
torch.save({
    'model_state': model.state_dict(),
    'embedding_dim': 512,
    'image_size': IMAGE_SIZE,
    'pretrained_initialization': 'vggface2',
    'fine_tuned_on': 'IMSFD',
    'unfreeze_modules': UNFREEZE_MODULES,
    'classes': classes,
    'class_to_idx': class_to_idx,
}, backbone_export_path)

with open(EXPORT_DIR / 'model_metadata.json', 'w') as f:
    json.dump({
        'architecture': 'InceptionResnetV1',
        'initial_weights': 'VGGFace2',
        'adaptation': 'partial_fine_tuning',
        'training_dataset': 'IMSFD',
        'training_objective': 'ArcFace + CrossEntropyLoss',
        'unfrozen_modules': UNFREEZE_MODULES,
        'embedding_dim': 512,
        'input_size': [3, IMAGE_SIZE, IMAGE_SIZE],
        'num_training_identities': len(classes),
    }, f, indent=2)

print('Backbone hasil fine-tuning:', backbone_export_path)

## 14. Ekstraksi embedding validation dan test

Semua gambar validation dan test diproses menggunakan model backbone hasil fine-tuning yang sama. Embedding dinormalisasi L2 sebelum cosine similarity dihitung.

In [ ]:
@torch.no_grad()
def extract_embeddings(loader, model):
    model.eval()
    embeddings_list, labels_list, paths_list = [], [], []
    for images, targets, paths_batch in tqdm(loader, desc='Ekstraksi embedding'):
        images = images.to(DEVICE, non_blocking=True)
        with autocast_context():
            embeddings = model(images)
        embeddings = F.normalize(embeddings.float(), p=2, dim=1)
        embeddings_list.append(embeddings.cpu().numpy())
        labels_list.extend(targets.numpy().tolist())
        paths_list.extend(list(paths_batch))
    return np.concatenate(embeddings_list), np.asarray(labels_list), paths_list

val_eval_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=USE_AMP,
                             persistent_workers=(NUM_WORKERS > 0))
test_eval_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=USE_AMP,
                              persistent_workers=(NUM_WORKERS > 0))

val_embeddings, val_labels, val_paths = extract_embeddings(val_eval_loader, model)
test_embeddings, test_labels, test_paths = extract_embeddings(test_eval_loader, model)
print('Validation embeddings:', val_embeddings.shape)
print('Test embeddings:', test_embeddings.shape)

## 15. Membentuk pasangan genuine/impostor dan menentukan threshold dari validation

- **Genuine pair:** dua foto dari identitas yang sama.
- **Impostor pair:** dua foto dari identitas berbeda.
- Similarity dihitung dengan dot product karena embedding sudah dinormalisasi L2.
- Threshold dipilih hanya dari validation, dengan mencari titik ketika FAR dan FRR paling dekat (estimasi EER).

In [ ]:
def make_verification_pairs(embeddings, labels, pairs_per_class=100, seed=123):
    rng = np.random.default_rng(seed)
    by_class = {int(c): np.where(labels == c)[0] for c in np.unique(labels)}
    eligible = {c: idxs for c, idxs in by_class.items() if len(idxs) >= 2}
    if len(eligible) < 2:
        raise ValueError('Diperlukan minimal dua identitas dengan dua foto masing-masing untuk verification pairs.')

    positive_pairs = []
    for class_id, idxs in eligible.items():
        possible = len(idxs) * (len(idxs) - 1) // 2
        target_n = min(int(pairs_per_class), possible)
        chosen = set()
        while len(chosen) < target_n:
            a, b = rng.choice(idxs, size=2, replace=False)
            chosen.add(tuple(sorted((int(a), int(b)))))
        positive_pairs.extend(chosen)

    target_negative = len(positive_pairs)
    class_ids = list(eligible.keys())
    negative_pairs = set()
    max_tries = max(10000, target_negative * 200)
    tries = 0
    while len(negative_pairs) < target_negative and tries < max_tries:
        c1, c2 = rng.choice(class_ids, size=2, replace=False)
        a = int(rng.choice(eligible[int(c1)]))
        b = int(rng.choice(eligible[int(c2)]))
        negative_pairs.add(tuple(sorted((a, b))))
        tries += 1
    if len(negative_pairs) < target_negative:
        raise RuntimeError('Tidak berhasil membuat jumlah impostor pairs yang seimbang; kurangi pairs_per_class.')

    pairs = positive_pairs + list(negative_pairs)
    y_true = np.array([1] * len(positive_pairs) + [0] * len(negative_pairs), dtype=np.int64)
    scores = np.array([float(np.dot(embeddings[a], embeddings[b])) for a, b in pairs], dtype=np.float32)
    return scores, y_true


def far_frr_at_threshold(scores, y_true, threshold):
    predictions = (scores >= threshold).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    far = fp / (fp + tn) if (fp + tn) else 0.0
    frr = fn / (fn + tp) if (fn + tp) else 0.0
    return far, frr

val_scores, val_pair_labels = make_verification_pairs(
    val_embeddings, val_labels, pairs_per_class=100, seed=SEED
)
unique_scores = np.unique(val_scores)
if len(unique_scores) == 1:
    raise RuntimeError('Semua skor validation sama; periksa embedding/model.')
threshold_candidates = np.concatenate([
    [unique_scores[0] - 1e-6],
    (unique_scores[:-1] + unique_scores[1:]) / 2.0,
    [unique_scores[-1] + 1e-6],
])

threshold_rows = []
for threshold in threshold_candidates:
    far, frr = far_frr_at_threshold(val_scores, val_pair_labels, float(threshold))
    threshold_rows.append({
        'threshold': float(threshold), 'FAR': float(far), 'FRR': float(frr),
        'absolute_FAR_FRR_gap': float(abs(far - frr)),
    })
threshold_df = pd.DataFrame(threshold_rows)
best_threshold_row = threshold_df.sort_values(
    ['absolute_FAR_FRR_gap', 'FAR', 'threshold'], ascending=[True, True, False]
).iloc[0]
VALIDATION_THRESHOLD = float(best_threshold_row['threshold'])
VALIDATION_EER_ESTIMATE = float((best_threshold_row['FAR'] + best_threshold_row['FRR']) / 2.0)

threshold_df.to_csv(OUTPUT_DIR / 'validation_threshold_search.csv', index=False)
print('Validation pairs:', len(val_pair_labels))
print('Threshold dipilih dari validation:', VALIDATION_THRESHOLD)
print('Validation FAR:', best_threshold_row['FAR'])
print('Validation FRR:', best_threshold_row['FRR'])
print('Validation EER estimate:', VALIDATION_EER_ESTIMATE)

## 16. Evaluasi final pada Testing resmi dengan threshold yang sudah dikunci

Threshold dari validation digunakan apa adanya. Kita **tidak mencari threshold baru dari test**. Test digunakan hanya untuk menghitung performa akhir.

In [ ]:
test_scores, test_pair_labels = make_verification_pairs(
    test_embeddings, test_labels, pairs_per_class=200, seed=SEED + 1
)
test_predictions = (test_scores >= VALIDATION_THRESHOLD).astype(np.int64)

tn, fp, fn, tp = confusion_matrix(test_pair_labels, test_predictions, labels=[0, 1]).ravel()
far = fp / (fp + tn) if (fp + tn) else 0.0
frr = fn / (fn + tp) if (fn + tp) else 0.0
metrics = {
    'threshold_selected_on_validation': VALIDATION_THRESHOLD,
    'validation_EER_estimate': VALIDATION_EER_ESTIMATE,
    'number_of_test_pairs': int(len(test_pair_labels)),
    'test_accuracy': float(accuracy_score(test_pair_labels, test_predictions)),
    'test_precision': float(precision_score(test_pair_labels, test_predictions, zero_division=0)),
    'test_recall': float(recall_score(test_pair_labels, test_predictions, zero_division=0)),
    'test_f1_score': float(f1_score(test_pair_labels, test_predictions, zero_division=0)),
    'test_FAR': float(far),
    'test_FRR': float(frr),
    'test_ROC_AUC': float(roc_auc_score(test_pair_labels, test_scores)),
    'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
}

with open(OUTPUT_DIR / 'verification_metrics_test.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

cm = confusion_matrix(test_pair_labels, test_predictions, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm)
ax.set_title('Confusion Matrix — Official Test Verification')
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_xticks([0, 1], ['Non-match', 'Match'])
ax.set_yticks([0, 1], ['Non-match', 'Match'])
for (row, col), value in np.ndenumerate(cm):
    ax.text(col, row, f'{value:,}', ha='center', va='center')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'confusion_matrix_verification_test.png', dpi=160)
plt.show()

fpr, tpr, roc_thresholds = roc_curve(test_pair_labels, test_scores)
plt.figure(figsize=(5.5, 4.5))
plt.plot(fpr, tpr, label=f"ROC AUC = {metrics['test_ROC_AUC']:.4f}")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Official Test Verification')
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig(PLOT_DIR / 'roc_curve_verification_test.png', dpi=160)
plt.show()

## 17. Daftar file output

Folder output baru: `MyDrive/IMSFD_ArcFace_Output_v2/`

```text
IMSFD_ArcFace_Output_v2/
├── 01_audit/
│   ├── dataset_manifest.csv
│   ├── identity_counts_by_split.csv
│   ├── subset_split_summary.csv
│   ├── format_summary.csv
│   ├── color_mode_summary.csv
│   ├── resolution_summary.csv
│   ├── corrupt_files.csv              (jika ada)
│   └── sample_training_faces.png
├── 02_splits/
│   ├── train_manifest.csv
│   ├── validation_manifest.csv
│   ├── test_manifest_used.csv
│   └── identity_mapping.csv
├── 03_checkpoints/
│   ├── best_checkpoint.pth
│   └── last_checkpoint.pth
├── 04_plots/
│   ├── loss_curve.png
│   ├── arcface_top1_curve.png
│   ├── confusion_matrix_verification_test.png
│   └── roc_curve_verification_test.png
├── 05_model_export/
│   ├── inceptionresnetv1_imsfd_partial_finetuned.pth
│   └── model_metadata.json
├── training_history.csv
├── validation_threshold_search.csv
└── verification_metrics_test.json
```

**Catatan evaluasi:** metrik verification bergantung pada protokol pasangan genuine/impostor, pembagian data, preprocessing, dan threshold validation. Jangan membandingkan angka dari protokol yang berbeda seolah-olah setara. Untuk data yang berasal dari frame video, pembagian berdasarkan sumber video/sesi lebih baik daripada pembagian acak per file; periksa metadata nama file jika tersedia.